### Matching

Match extracted resume skills with job description skills
* Calculate dictionary score
* Calculate jaccard score 

In [13]:
# Dictionary score
from importlib import reload
import sys
from pathlib import Path
import json
import pandas as pd

# Add ../src to import path
src_path = (Path.cwd().parent / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

reload(sys.modules["dictionary_match"])

from dictionary_match import compute_dictionary_pairs_and_topk, dumps_skill_columns
from dictionary_match import to_skill_list

processed_dir = "../data/processed"

In [9]:
# Use ranked extracted files if present, else fall back to raw files
resume_ranked_path = f"{processed_dir}/resume_skills_extracted.csv"
job_ranked_path = f"{processed_dir}/job_skills_extracted.csv"


resume_out = pd.read_csv(resume_ranked_path)
job_out = pd.read_csv(job_ranked_path)
print("Loaded ranked extracted files")


# Parse JSON-string skill columns back into lists if needed
for df in (resume_out, job_out):
    for col in ["extracted_skills", "extracted_skills_ranked"]:
        if col in df.columns:
            df[col] = df[col].apply(to_skill_list)

# Use ranked skills if present; otherwise fall back to raw extracted skills
resume_skill_col = "extracted_skills_ranked" if "extracted_skills_ranked" in resume_out.columns else "extracted_skills"
job_skill_col = "extracted_skills_ranked" if "extracted_skills_ranked" in job_out.columns else "extracted_skills"

# Compute dictionary pairs and top-K matches
# Each (resume, job) pair is scored based on skill overlap, and the top-K jobs are identified for each resume.
# 712 resumes x 3000 jobs = 2.136 million pairs
dictionary_pairs_df, dictionary_topk_df = compute_dictionary_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    resume_skill_col=resume_skill_col,
    job_skill_col=job_skill_col,
    top_k=5,
)

print("Pair rows:", dictionary_pairs_df.shape)
print("Top-K rows:", dictionary_topk_df.shape)
display(dictionary_topk_df.head(5))


Loaded ranked extracted files
Pair rows: (2136000, 14)
Top-K rows: (3560, 15)


,resume_id,resume_idx,resume_category,job_idx,job_link,job_title,company,job_location,dictionary_score,job_coverage,skill_jaccard,matched_skills,missing_skills,extra_skills,rank
0,20345168,0,ACCOUNTANT,452,https://www.linkedin.com/jobs/view/registered-...,Registered Nurse - Tele at Health Advocates Ne...,Health eCareers,"Cleveland, OH",35.16,0.5000,0.0055,[health],[health care],"[accountant, accounting duties, accounting ser...",1
1,20345168,0,ACCOUNTANT,1989,https://www.linkedin.com/jobs/view/customer-se...,Customer Service Representative,PickleballCentral.com,"Kent, WA",28.31,0.3889,0.0363,"[customer, customer service, excel, microsoft ...","[communication, computer, computer literacy, d...","[accountant, accounting duties, accounting ser...",2
2,20345168,0,ACCOUNTANT,638,https://www.linkedin.com/jobs/view/assistant-m...,Assistant Manager,HAZA Bell,"Hazleton, PA",27.19,0.3750,0.0312,"[customer, customer service, hiring, inventory...","[food, food safety, inventory management, labo...","[accountant, accounting duties, accounting ser...",3
3,20345168,0,ACCOUNTANT,2637,https://www.linkedin.com/jobs/view/managers-im...,Managers Immediate Openings,McDonald's,"Richmond Hill, GA",26.73,0.3750,0.0160,"[customer, customer service, team]","[hospitality, leadership, management leadershi...","[accountant, accounting duties, accounting ser...",4
4,20345168,0,ACCOUNTANT,1157,https://www.linkedin.com/jobs/view/restaurant-...,Restaurant Manager,Viva Margarita,"Edgewater, NJ",25.79,0.3571,0.0262,"[customer, customer service, organizational, o...","[adaptability, communication, leadership, mana...","[accountant, accounting duties, accounting ser...",5


In [10]:
# Save outputs for downstream fusion
pairs_save = dumps_skill_columns(dictionary_pairs_df, ["matched_skills", "missing_skills"])
topk_save = dumps_skill_columns(dictionary_topk_df, ["matched_skills", "missing_skills"])

pairs_path = f"{processed_dir}/dictionary_pair_scores.csv"
topk_path = f"{processed_dir}/dictionary_topk_per_resume.csv"
pairs_save.to_csv(pairs_path, index=False)
topk_save.to_csv(topk_path, index=False)

print("Saved:")
print(pairs_path)
print(topk_path)

Saved:
../data/processed/dictionary_pair_scores.csv
../data/processed/dictionary_topk_per_resume.csv
